[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/chatbot-external-memory.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239440-lesson-6-chatbot-w-summarizing-messages-and-external-memory)

# 具有消息摘要和外部数据库记忆的聊天机器人

## 回顾

我们已经学习了如何自定义图状态模式和reducer。
 
我们还展示了许多在图状态中修剪或过滤消息的技巧。

我们使用这些概念构建了一个具有记忆功能的聊天机器人，可以生成对话的运行摘要。

## 目标

但是，如果我们希望聊天机器人具有无限期持续的记忆怎么办？

现在，我们将介绍一些支持外部数据库的更高级的检查点保存器。

在这里，我们将展示如何使用[Sqlite作为检查点保存器](https://langchain-ai.github.io/langgraph/concepts/low_level/#checkpointer)，但其他检查点保存器，如[Postgres](https://langchain-ai.github.io/langgraph/how-tos/persistence_postgres/)也可以使用！

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph-checkpoint-sqlite langchain_core langgraph langchain_openai

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## Sqlite

这里一个好的起点是[SqliteSaver检查点保存器](https://langchain-ai.github.io/langgraph/concepts/low_level/#checkpointer)。

Sqlite是一个[小巧、快速、非常流行](https://x.com/karpathy/status/1819490455664685297)的SQL数据库。
 
如果我们提供`":memory:"`，它会创建一个内存中的Sqlite数据库。

In [ ]:
import sqlite3
# 内存数据库
conn = sqlite3.connect(":memory:", check_same_thread = False)

但是，如果我们提供数据库路径，那么它将为我们创建一个数据库！

In [ ]:
# 如果文件不存在则下载文件并连接到本地数据库
!mkdir -p state_db && [ ! -f state_db/example.db ] && wget -P state_db https://github.com/langchain-ai/langchain-academy/raw/main/module-2/state_db/example.db

db_path = "state_db/example.db"
conn = sqlite3.connect(db_path, check_same_thread=False)

In [ ]:
# 这是我们的检查点保存器
from langgraph.checkpoint.sqlite import SqliteSaver
memory = SqliteSaver(conn)

让我们重新定义我们的聊天机器人。

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

from langgraph.graph import END
from langgraph.graph import MessagesState

model = ChatOpenAI(model="gpt-4o",temperature=0)

class State(MessagesState):
    summary: str

# 定义调用模型的逻辑
def call_model(state: State):
    
    # 获取摘要（如果存在）
    summary = state.get("summary", "")

    # 如果有摘要，我们添加它
    if summary:
        
        # 将摘要添加到系统消息
        system_message = f"更早对话的摘要: {summary}"

        # 将摘要附加到任何较新的消息
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages)
    return {"messages": response}

def summarize_conversation(state: State):
    
    # 首先，我们获取任何现有的摘要
    summary = state.get("summary", "")

    # 创建我们的摘要提示
    if summary:
        
        # 摘要已经存在
        summary_message = (
            f"这是到目前为止对话的摘要: {summary}\n\n"
            "通过考虑上面的新消息来扩展摘要:"
        )
        
    else:
        summary_message = "创建上述对话的摘要:"

    # 将提示添加到我们的历史记录
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 删除除最近2条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

# 确定是否结束或摘要对话
def should_continue(state: State):
    
    """返回要执行的下一个节点。"""
    
    messages = state["messages"]
    
    # 如果有超过六条消息，那么我们摘要对话
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否则我们可以直接结束
    return END

现在，我们只需使用sqlite检查点保存器重新编译。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START

# 定义一个新的图
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 设置入口点为conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 编译
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

现在，我们可以多次调用图。

In [ ]:
# 创建一个线程
config = {"configurable": {"thread_id": "1"}}

# 开始对话
input_message = HumanMessage(content="你好！我是Lance")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="我的名字是什么？")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="我喜欢49人队！")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

让我们确认状态保存在本地。

In [ ]:
config = {"configurable": {"thread_id": "1"}}
graph_state = graph.get_state(config)
graph_state

### 持久化状态

使用像Sqlite这样的数据库意味着状态是持久化的！

例如，我们可以重启notebook内核，并看到我们仍然可以从磁盘上的Sqlite数据库加载状态。

In [ ]:
# 创建一个线程
config = {"configurable": {"thread_id": "1"}}
graph_state = graph.get_state(config)
graph_state

## LangGraph Studio

**⚠️ 免责声明**

自这些视频录制以来，我们已经更新了Studio，使其可以在本地运行并在浏览器中打开。这现在是运行Studio的首选方式（而不是像视频中显示的那样使用桌面应用程序）。关于本地开发服务器的文档请看[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)，关于本地Studio的文档请看[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)。要启动本地开发服务器，请在此模块的`/studio`目录中的终端中运行以下命令：

```
langgraph dev
```

您应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。